# DS/CMPSC 410 MiniProject Deliverable #3

# Spring 2025
### Instructor: Prof. John Yen
### TA: Peng Jin and Jingxi Zhu

### Learning Objectives
- Be able to apply PCA to reduce the high dimensional feature space to facilitate ML for high dimensional data.
- Like Minproject Deliverable #2, focus on the clustering of non-extreme multi-port scanners based on the 120 top ports they scanned.
- Be able to reduce the high dimensionality of One-Hot-Encoded features using Principal Component Analysis (PCA)
- Be able to perform k-means with and without dimension reduction using PCA, and compare the results using silhouette score and mirai external labels.
- Be able to obtain cluster centers of the original feature space for clustering results using PCA and k-means.
- After successful clustering of the small Darknet dataset (with and without dimension reduction using PCA), conduct clustering on the large Darknet dataset in the cluster mode.
- Compare the clustering results of the large dataset with and without dimension reduction using PCA.

## Submit the following items:
- Successfully completed Jupyter Notebook (run in local mode), in html format.
- Items for cluster mode (Exercise 8):
- - Submit the .py file  (5 points)
- - Submit the the log file that contains the run time information for a successful execution in the cluster mode. (5 points)
- - Submit the output file that records the cluster summary in the cluster mode (without PCA) (10 points)
- - Submit the output file that records the cluster summary in the cluster mode (with PCA) (10 points)
- - Discuss the Silihouette score and Mirai ratio of clusters generated by k-means clustering with PCA and without PCA (in a separate word document) (10 points)

### Total points: 100 
- Exercise 1: 1 point
- Exercise 2: 9 points 
- Exercise 3: 5 points 
- Exercise 4: 5 points
- Exercise 5: 5 points
- Exercise 6: 10 points
- Exercise 7: 25 points
- Exercise 8: 40 points
  
### Due: 11:59 pm, April 20th, 2025
### Early Submission bonus (before midnight April 13th): 10 points

In [26]:
import pyspark
import csv

In [27]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, LongType
from pyspark.sql.functions import col, column
from pyspark.sql.functions import expr
from pyspark.sql.functions import split
from pyspark.sql.functions import array_contains
from pyspark.sql import Row
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler, IndexToString
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA

In [28]:
ss = SparkSession.builder.master("local").appName("MiniProject 3 PCA for k-meas Clustering using 120 OHE").getOrCreate()

In [29]:
ss.sparkContext.setLogLevel("WARN")

# We can use the header of data files to infer schema.

## Exercise 1 (1 point)
Complete the path for input file in the code below and enter your name in this Markdown cell:
- Name: Aidan Vesci
### Note: You will need to change the name of the input file in the cluster mode to `Day_2020_profile.csv`

In [5]:
Scanners_df = ss.read.csv("/storage/home/ajv5723/work/MiniProj3/sampled_profile.csv", header= True, inferSchema=True )

## We can use printSchema() to display the schema of the DataFrame Scanners_df to see whether it was inferred correctly.

In [30]:
Scanners_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- numports: integer (nullable = true)
 |-- lifetime: double (nullable = true)
 |-- Bytes: integer (nullable = true)
 |-- Packets: integer (nullable = true)
 |-- average_packetsize: integer (nullable = true)
 |-- MinUniqueDests: integer (nullable = true)
 |-- MaxUniqueDests: integer (nullable = true)
 |-- MinUniqueDest24s: integer (nullable = true)
 |-- MaxUniqueDest24s: integer (nullable = true)
 |-- average_lifetime: double (nullable = true)
 |-- mirai: boolean (nullable = true)
 |-- zmap: boolean (nullable = true)
 |-- masscan: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_types_scanned_str: string (nullable = true)
 |-- ports_scanned_str: string (nullable = true)
 |-- host_tags_per_censys: string (nullable = true)
 |-- host_services_per_censys: string (nullable = true)



# In this mini project, our goal is to implement, interpret, and evaluate the results for reducing the dimensionality of the feature space using PCA.

## Like MiniProject 2, we can filter out those scanners that scan only 1 port, because they can be easily grouped using groupBy on ``ports_scanned_str``

### Because the feature `numports` record the total number of ports being scanned by each scanner, we can use it to separate 1-port-scanners from multi-port-scanners.

In [31]:
multi_port_scanners= Scanners_df.where(col("numports")>1)

In [32]:
multi_port_scanners_count = multi_port_scanners.count()

In [33]:
print(multi_port_scanners_count)

73663


# Like Miniproject 2, we select a threashold for extreme scanners based on the largest number of ports that are scanned by at least two scanners

In [34]:
ScannersCount_byNumPorts = multi_port_scanners.groupby("numports").count()

In [35]:
ScannersCount_byNumPorts.show(3)

+--------+-----+
|numports|count|
+--------+-----+
|    1238|    1|
|      31|   33|
|   31161|    1|
+--------+-----+
only showing top 3 rows



# Exercise 2 (9 points)
Complete the code below to find Non-extreme Multi-port scanners, using ``ScannersCount_byNumPorts`` DataFrame, by following the following steps:
- Step 1: Find the largest ``numports`` among all scanners that scann at least two ports.  We can use DataFrame aggregation method ``agg({ "numports" : "max" })``.  The result is a DataFrame with only column named as ``max(numports)``.  Obtain the value from the DataFrame and save it as a threshold for extreme scanners.
- Step 2: Filter ``multi_port_scanners`` DataFrame for those who are below the threshold for extreme scanners, save the result in ``non_extreme_multi_port_scanners`` DataFrame, which will be the data we want to cluster using k-means, with and without PCA.
- Step 3: Save the scanners whose ``numports`` are above the threshold for extreme scanners in a CSV file.

## Step 1

In [36]:
Scanners_not_unique_numports =  ScannersCount_byNumPorts.where( col("count") > 1) 

In [37]:
ExtremeScannersNumports_thresholdDF = Scanners_not_unique_numports.agg({ "numports" : "max" })

In [38]:
ExtremeScannersNumports_thresholdDF.show()

+-------------+
|max(numports)|
+-------------+
|          654|
+-------------+



In [39]:
max_non_rare_NumPorts_rdd = ExtremeScannersNumports_thresholdDF.rdd.map(lambda x: x[0])
max_non_rare_NumPorts_rdd.take(1)

[654]

In [40]:
max_non_rare_NumPorts_list = max_non_rare_NumPorts_rdd.collect()
print(max_non_rare_NumPorts_list)

[654]


In [41]:
max_non_rare_NumPorts=max_non_rare_NumPorts_list[0]
print(max_non_rare_NumPorts)

654


## Step 2

In [42]:
non_extreme_multi_port_scanners = Scanners_df.where(col("numports") <= max_non_rare_NumPorts).where(col("numports") > 1)

In [43]:
non_extreme_multi_port_scanners.count()

73599

## Step 3

In [44]:
extreme_scanners = Scanners_df.where(col("numports") > max_non_rare_NumPorts)

In [27]:
path2="/storage/home/ajv5723/work/MiniProj3/local/Extreme_Scanners.csv"
extreme_scanners.write.option("header",True).csv(path2)

25/04/07 18:32:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj3/sampled_profile.csv


# Part A: One Hot Encoding of Top 120 Ports
- Like Miniproject 2, we want to apply one hot encoding to the top 120 ports scanned by scanners.  
- Unlike Miniproject 2, however, we will apply PCA to reduce the dimensionality (from 120 port features to 30 PCA features )

In [45]:
non_extreme_multi_port_scanners.select("ports_scanned_str").show(4)

+--------------------+
|   ports_scanned_str|
+--------------------+
|         17128-17136|
|17128-17130-17132...|
|23-80-81-1023-232...|
|17128-17132-17136...|
+--------------------+
only showing top 4 rows



# For each port scanned, count the Total Number of Scanners that Scan the Given Port
Like MiniProject 2, to calculate this, we need to 
- (a) convert the ports_scanned_str into an array/list of ports
- (b) Convert the DataFrame into an RDD
- (c) Use flatMap to count the total number of scanners for each port.

## (a) Split the column "Ports_Array" into an Array of ports.

In [46]:
# (a)
NEMP_Scanners_df=non_extreme_multi_port_scanners.withColumn("Ports_Array", split(col("ports_scanned_str"), "-") )
NEMP_Scanners_df.show(2)

+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+
|    _c0|     id|numports|lifetime|Bytes|Packets|average_packetsize|MinUniqueDests|MaxUniqueDests|MinUniqueDest24s|MaxUniqueDest24s|average_lifetime|mirai| zmap|masscan|country|traffic_types_scanned_str|   ports_scanned_str|host_tags_per_censys|host_services_per_censys|         Ports_Array|
+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+
|2091467|2091467|       2|  199.84|  752|     12|                62|             1|             1|               1|         

25/04/07 18:55:18 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj3/sampled_profile.csv


## (b) We convert the column ```Ports_Array``` into an RDD so that we can apply flatMap for counting.

In [47]:
Ports_Scanned_RDD = NEMP_Scanners_df.select("Ports_Array").rdd

In [48]:
Ports_Scanned_RDD.take(2)

[Row(Ports_Array=['17128', '17136']),
 Row(Ports_Array=['17128', '17130', '17132', '17134', '17136', '17138', '17140'])]

## (c) Like Miniproject 2, we can count the total number of scanners for each port by counting the total occurance of each port number through flatMap.
### We can then count the total number of occurance of a port using map and reduceByKey, like counting word/hashtag frequency in tweets.

In [49]:
Ports_Scanned_RDD.take(3)

[Row(Ports_Array=['17128', '17136']),
 Row(Ports_Array=['17128', '17130', '17132', '17134', '17136', '17138', '17140']),
 Row(Ports_Array=['23', '80', '81', '1023', '2323', '5555', '7574', '8080', '8443', '37215', '49152', '52869'])]

In [50]:
Ports_list_RDD = Ports_Scanned_RDD.flatMap(lambda row: row.Ports_Array )

In [51]:
Ports_list_RDD.take(3)

['17128', '17136', '17128']

In [52]:
Port_1_RDD = Ports_list_RDD.map(lambda x: (x, 1))
Port_1_RDD.take(2)

[('17128', 1), ('17136', 1)]

In [53]:
Port_count_RDD = Port_1_RDD.reduceByKey(lambda x,y: x+y, 5)
Port_count_RDD.take(3)

[('17128', 25074), ('17136', 24928), ('17134', 18183)]

In [54]:
Port_count_RDD.count()

47931

## Exercise 3 (5 points) Complete the code below to finds top 120 ports scanned by non-extreme multi-port scanners. We use top 120 ports for Mini-project 3 (both local mode and cluster mode).

In [55]:
Sorted_Count_Port_RDD = Port_count_RDD.map(lambda x: (x[1], x[0])).sortByKey( ascending = False)

In [56]:
top_k_ports = 120

In [57]:
Sorted_Ports_RDD= Sorted_Count_Port_RDD.map(lambda x: x[1] )
Top_Ports_list = Sorted_Ports_RDD.take(top_k_ports)

In [58]:
Top_Ports_list

['17132',
 '17130',
 '17140',
 '17128',
 '17138',
 '17136',
 '17134',
 '17142',
 '80',
 '8080',
 '23',
 '2323',
 '81',
 '1023',
 '5555',
 '52869',
 '8443',
 '49152',
 '7574',
 '37215',
 '54594',
 '34218',
 '34220',
 '33962',
 '33968',
 '34224',
 '34228',
 '33960',
 '33964',
 '34216',
 '33970',
 '34226',
 '33972',
 '50401',
 '34222',
 '34230',
 '33966',
 '33974',
 '445',
 '0',
 '22',
 '8291',
 '8728',
 '1433',
 '8000',
 '8081',
 '5353',
 '2004',
 '11211',
 '6881',
 '443',
 '8082',
 '4000',
 '5060',
 '8083',
 '8088',
 '6379',
 '9527',
 '30301',
 '7001',
 '9200',
 '7002',
 '1027',
 '1900',
 '3389',
 '5900',
 '21',
 '6380',
 '88',
 '35',
 '8181',
 '5000',
 '389',
 '56880',
 '5001',
 '137',
 '8008',
 '7547',
 '49153',
 '4444',
 '139',
 '2222',
 '8001',
 '3544',
 '8888',
 '5984',
 '2480',
 '53',
 '1883',
 '873',
 '631',
 '9000',
 '50070',
 '161',
 '4786',
 '60001',
 '8090',
 '27017',
 '85',
 '12866',
 '3443',
 '111',
 '83',
 '82',
 '548',
 '5061',
 '995',
 '5901',
 '554',
 '10001',
 '9943',


#  A.2 One Hot Encoding of Top K Ports
## One-Hot-Encoded Feature/Column Name
Like Mini Project 2, we need to create a name for each one-hot-encoded feature. We adopt the convention that the column name for each top k port is "PortXXXX", where "XXXX" is a port number. This can be done by concatenating "Port" with a port number (string) in the sorted list ``Top_Ports_list`` using ``+``.

The code below is an example of OHE feature name for the last top_k_ports (i.e., the 120th top k port).

In [59]:
Top_Ports_list[top_k_ports - 1]

'9999'

In [60]:
FeatureName = "Port"+Top_Ports_list[top_k_ports - 1]

In [61]:
FeatureName

'Port9999'

## One-Hot-Encoding using withColumn and array_contains

In [62]:
from pyspark.sql.functions import array_contains

## Similar to MiniProject 2, generate Hot-One Encoded Feature for each of the top k ports in the Top_Ports_list

- Iterate through the Top_Ports_list so that each top port is one-hot encoded into the DataFrame for non-extreme multi-port scanners (i.e., `NEMP_Scanners2.df`).

## Exercise 4 (5 points) Complete the following PySpark code for encoding the n top ports using One Hot Encoding, where n is specified by the variable ```top_k_ports```

In [63]:
top_k_ports

120

In [64]:
Top_Ports_list[top_k_ports - 1]

'9999'

In [65]:
# Initialize NEMP_Scanners2_df
NEMP_Scanners2_df = NEMP_Scanners_df
NEMP_Scanners_df.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>]

In [66]:
for i in range(0, top_k_ports):
    # "Port" + Top_Ports_list[i]  is the name of each new feature created through One Hot Encoding Top_Ports_list
    NEMP_Scanners3_df = NEMP_Scanners2_df.withColumn("Port" + Top_Ports_list[i], array_contains("Ports_Array", Top_Ports_list[i]))
    NEMP_Scanners2_df = NEMP_Scanners3_df

In [67]:
NEMP_Scanners2_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- numports: integer (nullable = true)
 |-- lifetime: double (nullable = true)
 |-- Bytes: integer (nullable = true)
 |-- Packets: integer (nullable = true)
 |-- average_packetsize: integer (nullable = true)
 |-- MinUniqueDests: integer (nullable = true)
 |-- MaxUniqueDests: integer (nullable = true)
 |-- MinUniqueDest24s: integer (nullable = true)
 |-- MaxUniqueDest24s: integer (nullable = true)
 |-- average_lifetime: double (nullable = true)
 |-- mirai: boolean (nullable = true)
 |-- zmap: boolean (nullable = true)
 |-- masscan: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_types_scanned_str: string (nullable = true)
 |-- ports_scanned_str: string (nullable = true)
 |-- host_tags_per_censys: string (nullable = true)
 |-- host_services_per_censys: string (nullable = true)
 |-- Ports_Array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Port17132: 

## Exercise 5 (5 points)  Complete the code below to use k-means to cluster non-extreme multi-port scanners using one-hot-encoded top 120 ports.

# Specify One-Hot Encoded Top k Ports as Input Features for k Means Clustering

In [68]:
input_features = [ ]
for i in range(0, top_k_ports):
    input_features.append("Port" + Top_Ports_list[i])

In [69]:
print(input_features)

['Port17132', 'Port17130', 'Port17140', 'Port17128', 'Port17138', 'Port17136', 'Port17134', 'Port17142', 'Port80', 'Port8080', 'Port23', 'Port2323', 'Port81', 'Port1023', 'Port5555', 'Port52869', 'Port8443', 'Port49152', 'Port7574', 'Port37215', 'Port54594', 'Port34218', 'Port34220', 'Port33962', 'Port33968', 'Port34224', 'Port34228', 'Port33960', 'Port33964', 'Port34216', 'Port33970', 'Port34226', 'Port33972', 'Port50401', 'Port34222', 'Port34230', 'Port33966', 'Port33974', 'Port445', 'Port0', 'Port22', 'Port8291', 'Port8728', 'Port1433', 'Port8000', 'Port8081', 'Port5353', 'Port2004', 'Port11211', 'Port6881', 'Port443', 'Port8082', 'Port4000', 'Port5060', 'Port8083', 'Port8088', 'Port6379', 'Port9527', 'Port30301', 'Port7001', 'Port9200', 'Port7002', 'Port1027', 'Port1900', 'Port3389', 'Port5900', 'Port21', 'Port6380', 'Port88', 'Port35', 'Port8181', 'Port5000', 'Port389', 'Port56880', 'Port5001', 'Port137', 'Port8008', 'Port7547', 'Port49153', 'Port4444', 'Port139', 'Port2222', 'Por

# Part B k-Means Clustering using 120 OHE features (number of clusters k=200)

In [70]:
va = VectorAssembler().setInputCols(input_features).setOutputCol("features")

In [71]:
NEMP_Scanners2_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- numports: integer (nullable = true)
 |-- lifetime: double (nullable = true)
 |-- Bytes: integer (nullable = true)
 |-- Packets: integer (nullable = true)
 |-- average_packetsize: integer (nullable = true)
 |-- MinUniqueDests: integer (nullable = true)
 |-- MaxUniqueDests: integer (nullable = true)
 |-- MinUniqueDest24s: integer (nullable = true)
 |-- MaxUniqueDest24s: integer (nullable = true)
 |-- average_lifetime: double (nullable = true)
 |-- mirai: boolean (nullable = true)
 |-- zmap: boolean (nullable = true)
 |-- masscan: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_types_scanned_str: string (nullable = true)
 |-- ports_scanned_str: string (nullable = true)
 |-- host_tags_per_censys: string (nullable = true)
 |-- host_services_per_censys: string (nullable = true)
 |-- Ports_Array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Port17132: 

In [72]:
data= va.transform(NEMP_Scanners2_df)

In [73]:
data.show(3)

25/04/07 18:56:09 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/04/07 18:56:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj3/sampled_profile.csv


+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+---------+---------+---------+---------+---------+---------+---------+---------+------+--------+------+--------+------+--------+--------+---------+--------+---------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------+-----+------+--------+--------+--------+--------+--------+--------+--------+---------+--------+-------+--------+--------+--------+--------+--------+--------+--------+---------+--------+--------+--------+--------+--------+--------+--------+------+--------+------+------+--------+--------+-------+---------+--------+-------+--------+-----

In [74]:
data.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17132: boolean, Port17130: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boolean, Port3396

In [75]:
total_clusters = 200
km = KMeans(featuresCol= "features", predictionCol="prediction").setK(total_clusters).setSeed(123)
km.explainParams()

'distanceMeasure: the distance measure. Supported options: \'euclidean\' and \'cosine\'. (default: euclidean)\nfeaturesCol: features column name. (default: features, current: features)\ninitMode: The initialization algorithm. This can be either "random" to choose random points as initial cluster centers, or "k-means||" to use a parallel variant of k-means++ (default: k-means||)\ninitSteps: The number of steps for k-means|| initialization mode. Must be > 0. (default: 2)\nk: The number of clusters to create. Must be > 1. (default: 2, current: 200)\nmaxIter: max number of iterations (>= 0). (default: 20)\npredictionCol: prediction column name. (default: prediction, current: prediction)\nseed: random seed. (default: -2819157463477060340, current: 123)\ntol: the convergence tolerance for iterative algorithms (>= 0). (default: 0.0001)\nweightCol: weight column name. If this is not set or empty, we treat all instance weights as 1.0. (undefined)'

In [76]:
kmModel=km.fit(data)

In [77]:
NEMP_Scanners_df.unpersist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>]

In [78]:
kmModel

KMeansModel: uid=KMeans_a3cc3b8325f2, k=200, distanceMeasure=euclidean, numFeatures=120

In [79]:
predictions = kmModel.transform(data)

In [80]:
predictions.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17132: boolean, Port17130: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boolean, Port3396

In [81]:
predictions.show(1)

+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+-----------------+--------------------+------------------------+--------------+---------+---------+---------+---------+---------+---------+---------+---------+------+--------+------+--------+------+--------+--------+---------+--------+---------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------+-----+------+--------+--------+--------+--------+--------+--------+--------+---------+--------+-------+--------+--------+--------+--------+--------+--------+--------+---------+--------+--------+--------+--------+--------+--------+--------+------+--------+------+------+--------+--------+-------+---------+--------+-------+--------+--------+-----

In [82]:
Cluster1_df=predictions.where(col("prediction")==0)

In [83]:
Cluster1_df.count()

697

In [84]:
summary = kmModel.summary

In [85]:
summary.clusterSizes

[697,
 2319,
 1681,
 1221,
 323,
 447,
 1337,
 95,
 1137,
 133,
 932,
 7185,
 47,
 969,
 345,
 457,
 454,
 198,
 1112,
 100,
 118,
 302,
 141,
 220,
 480,
 44,
 381,
 220,
 594,
 411,
 140,
 569,
 376,
 424,
 169,
 102,
 854,
 988,
 299,
 122,
 146,
 96,
 845,
 477,
 463,
 191,
 274,
 41,
 306,
 499,
 901,
 120,
 120,
 1225,
 268,
 116,
 56,
 126,
 117,
 476,
 311,
 70,
 205,
 38,
 214,
 256,
 107,
 60,
 134,
 173,
 362,
 100,
 90,
 103,
 290,
 65,
 153,
 105,
 357,
 6614,
 275,
 238,
 8,
 132,
 241,
 323,
 85,
 74,
 81,
 642,
 194,
 522,
 508,
 28,
 210,
 449,
 124,
 6,
 166,
 266,
 152,
 665,
 123,
 341,
 790,
 128,
 84,
 301,
 143,
 34,
 201,
 344,
 152,
 333,
 208,
 169,
 106,
 68,
 407,
 206,
 91,
 49,
 54,
 661,
 74,
 288,
 353,
 65,
 102,
 194,
 138,
 125,
 244,
 339,
 380,
 91,
 84,
 112,
 625,
 73,
 168,
 99,
 665,
 255,
 190,
 260,
 223,
 133,
 317,
 37,
 401,
 870,
 198,
 240,
 1004,
 196,
 278,
 324,
 137,
 544,
 157,
 121,
 211,
 144,
 165,
 383,
 696,
 77,
 75,
 90,
 119,

In [86]:
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)

In [87]:
print('Silhouette Score of the Clustering Result using 100 OHE Top Ports, without using PCA, is ', silhouette)

Silhouette Score of the Clustering Result using 100 OHE Top Ports, without using PCA, is  0.4994049639415963


In [88]:
centers = kmModel.clusterCenters()

# We will later use this two dimensional arrays ``centers[i][j]`` to access the cluster center for each port (jth top port) for cluster (i) generated by clustering using the k-means model.

In [89]:
print(centers)

[array([0.        , 0.80631277, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.65710187, 0.        , 0.        ,
       0.00143472, 0.        , 0.        , 0.        , 0.00430416,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00286944, 0.00430416, 0.04734577, 0.05164993, 0.05021521,
       0.03299857, 0.04447633, 0.00573888, 0.04591105, 0.        ,
       0.00286944, 0.00573888, 0.00143472, 0.        , 0.01721664,
       0.01004304, 0.00430416, 0.00143472, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00143472, 0.        , 0.        , 0.        ,
       0.        , 0.00143472, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00286944, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.    

# Similar to Miniproject 2, record cluster index, cluster size, percentage of Mirai scanners, and cluster centers for each clusters formed.
## The value of cluster center for a OHE top port is the percentage of data/clusters in the cluster that scans the top port. For example, a cluster center `[0.094, 0.8, 0, ...]` indicates the following
- 9.4% of the scanners in the cluster scan Top_Ports_list[0]: port 17132
- 80% of the scanners in the cluster scan Top_Ports_list[1]: port 17130
- No scanners in the cluster scan Top_Ports_list[2]: port 17140

# Exercise 6 (10 points) Complete the code below for computing the percentage of Mirai scanners for each scanner, and record it together with cluster centers for each cluster (without PCA).

In [90]:
import pandas as pd
import numpy as np
import math

In [91]:
c1_centers = centers[0][0:top_k_ports]

In [92]:
print(c1_centers)

[0.         0.80631277 0.         0.         0.         0.
 0.         0.65710187 0.         0.         0.00143472 0.
 0.         0.         0.00430416 0.         0.         0.
 0.         0.         0.00286944 0.00430416 0.04734577 0.05164993
 0.05021521 0.03299857 0.04447633 0.00573888 0.04591105 0.
 0.00286944 0.00573888 0.00143472 0.         0.01721664 0.01004304
 0.00430416 0.00143472 0.         0.         0.         0.
 0.         0.         0.         0.         0.00143472 0.
 0.         0.         0.         0.00143472 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.00286944 0.         0.         0.         0.         0.
 0.         0.         0.         0.00286944 0.         0.
 0.         0.         0.         0.         0.         0.00143472
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.       

In [93]:
# Define columns of the Pandas dataframe
column_list = ['cluster ID', 'size', 'mirai_ratio' ]
for feature in input_features:
    column_list.append(feature)
clusters_summary_df = pd.DataFrame( columns = column_list )
for i in range(0, total_clusters):
    cluster_i = predictions.where(col("prediction")==i)
    cluster_i.persist()
    cluster_i_size = cluster_i.count()
    cluster_i_mirai_count = cluster_i.where(col("mirai")).count()
    cluster_i_mirai_ratio = cluster_i_mirai_count/cluster_i_size
    if cluster_i_mirai_count > 0:
        print("Cluster ", i, "; Mirai Ratio:", cluster_i_mirai_ratio, "; Cluster Size: ", cluster_i_size)
    cluster_row = [i, cluster_i_size, cluster_i_mirai_ratio]
    for j in range(0, len(input_features)):
        cluster_row.append(centers[i][j] )
    clusters_summary_df.loc[i]= cluster_row
    cluster_i.unpersist()

Cluster  6 ; Mirai Ratio: 0.8339566192969334 ; Cluster Size:  1337
Cluster  11 ; Mirai Ratio: 0.0009742519137091162 ; Cluster Size:  7185
Cluster  19 ; Mirai Ratio: 0.22 ; Cluster Size:  100
Cluster  36 ; Mirai Ratio: 0.00117096018735363 ; Cluster Size:  854
Cluster  42 ; Mirai Ratio: 0.010650887573964497 ; Cluster Size:  845
Cluster  45 ; Mirai Ratio: 0.03664921465968586 ; Cluster Size:  191
Cluster  55 ; Mirai Ratio: 0.3706896551724138 ; Cluster Size:  116
Cluster  67 ; Mirai Ratio: 0.5833333333333334 ; Cluster Size:  60
Cluster  79 ; Mirai Ratio: 0.0007559721802237678 ; Cluster Size:  6614
Cluster  83 ; Mirai Ratio: 0.022727272727272728 ; Cluster Size:  132
Cluster  103 ; Mirai Ratio: 0.25806451612903225 ; Cluster Size:  341


In [94]:
path4= "/storage/home/ajv5723/work/MiniProj3/local/Clusters_Mirai_Ratio_120OHE_k200.csv"
clusters_summary_df.to_csv(path4, header=True)

In [95]:
predictions.unpersist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17132: boolean, Port17130: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boolean, Port3396

# Part C Using PCA for Dimension Reduction

# We use PCA to reduce the input dimension from 120 to 30.

In [96]:
reduced_dimension = 30
pca_model = PCA(k= reduced_dimension, inputCol = "features", outputCol="pca_features")

# The `pca_model` is a template for constructing a PCA model.
## After we apply `fit` to a PCA template, we obtain an actual mapping from the original feature space to the PCA space.

In [97]:
p_model = pca_model.fit(data)

25/04/07 19:00:30 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
25/04/07 19:00:30 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


In [98]:
p_data = p_model.transform(data)

In [99]:
p_data.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17132: boolean, Port17130: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boolean, Port3396

## Notice we change the `featuresCol` for k-means clustering to `"pca_features"` because we want to use its reduced dimension for clustering.

In [100]:
total_clusters = 200
km2 = KMeans(featuresCol= "pca_features", predictionCol="pca_prediction").setK(total_clusters).setSeed(123)
km2.explainParams()

'distanceMeasure: the distance measure. Supported options: \'euclidean\' and \'cosine\'. (default: euclidean)\nfeaturesCol: features column name. (default: features, current: pca_features)\ninitMode: The initialization algorithm. This can be either "random" to choose random points as initial cluster centers, or "k-means||" to use a parallel variant of k-means++ (default: k-means||)\ninitSteps: The number of steps for k-means|| initialization mode. Must be > 0. (default: 2)\nk: The number of clusters to create. Must be > 1. (default: 2, current: 200)\nmaxIter: max number of iterations (>= 0). (default: 20)\npredictionCol: prediction column name. (default: prediction, current: pca_prediction)\nseed: random seed. (default: -2819157463477060340, current: 123)\ntol: the convergence tolerance for iterative algorithms (>= 0). (default: 0.0001)\nweightCol: weight column name. If this is not set or empty, we treat all instance weights as 1.0. (undefined)'

In [101]:
data.unpersist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17132: boolean, Port17130: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boolean, Port3396

In [102]:
kmModel_p=km2.fit(p_data)

25/04/07 19:00:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj3/sampled_profile.csv


In [103]:
kmModel_p

KMeansModel: uid=KMeans_a3011dd2bc94, k=200, distanceMeasure=euclidean, numFeatures=30

In [104]:
p_predictions = kmModel_p.transform(p_data)

In [105]:
p_predictions.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: double, Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: double, mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17132: boolean, Port17130: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boolean, Port3396

In [106]:
summary_p = kmModel_p.summary

In [107]:
summary_p.clusterSizes

[410,
 53,
 1433,
 377,
 375,
 336,
 114,
 1207,
 109,
 96,
 7003,
 243,
 2324,
 877,
 665,
 232,
 998,
 454,
 114,
 203,
 213,
 134,
 165,
 85,
 426,
 222,
 132,
 375,
 611,
 7540,
 222,
 124,
 121,
 1003,
 666,
 56,
 109,
 699,
 440,
 119,
 323,
 23,
 60,
 1266,
 109,
 191,
 138,
 45,
 223,
 77,
 256,
 65,
 116,
 344,
 541,
 98,
 119,
 75,
 132,
 525,
 63,
 113,
 60,
 276,
 127,
 211,
 320,
 811,
 78,
 123,
 189,
 77,
 563,
 71,
 83,
 176,
 306,
 123,
 254,
 735,
 381,
 421,
 520,
 412,
 416,
 190,
 270,
 267,
 209,
 270,
 87,
 481,
 107,
 98,
 85,
 636,
 40,
 429,
 663,
 443,
 102,
 375,
 259,
 66,
 782,
 441,
 122,
 209,
 1004,
 194,
 246,
 555,
 58,
 323,
 57,
 264,
 254,
 139,
 79,
 69,
 142,
 253,
 207,
 41,
 282,
 324,
 62,
 94,
 439,
 108,
 65,
 82,
 26,
 373,
 432,
 774,
 45,
 693,
 709,
 29,
 87,
 79,
 270,
 683,
 187,
 58,
 41,
 286,
 130,
 306,
 87,
 386,
 216,
 114,
 854,
 316,
 202,
 72,
 196,
 321,
 570,
 274,
 230,
 62,
 177,
 297,
 121,
 102,
 411,
 329,
 68,
 350,
 5

# Notice that we need to specify `featuresCol` explicitly because the default is "features", but we are using "pca_features" as input features for Part C.

In [108]:
evaluator = ClusteringEvaluator(predictionCol="pca_prediction", featuresCol="pca_features")
silhouette = evaluator.evaluate(p_predictions)

In [109]:
print('Silhouette Score of the Clustering Result Using PCA-reduced dimension ',reduced_dimension,' on ', top_k_ports, ' OHE port features is ', silhouette)

Silhouette Score of the Clustering Result Using PCA-reduced dimension  30  on  120  OHE port features is  0.5332607661140135


In [110]:
top_k_ports

120

# Compute the Cluster Centers from clustering on PCA-reduced dimensions in the original dimensions (One Hot Encoded top port features)

## Record the cluster centers in the ``cluster_center_array`` where the row index refers to clusters, and the column index refers to OHE features (i.e., top ports).

In [111]:
import numpy as np
# Initialize the cluster center array (for the original port dimensions) to zeros
cluster_center_array = np.zeros([total_clusters, top_k_ports])

## For each OHE top port (say top_port_i), do the following:
- Step 1: Filter the p_prediction DataFrame on the OHE top port feature, which returns all scanners that scan the specific top port.
- Step 2: Apply ``groupBy`` on``pca_prediction``, which groups the filtered scanners by the cluster they belong to (based on clustering on PCA-reduced dimensions).
- Step 3: Apply ``count()`` to the DataFrame returned by ``groupBy``, which computes the total number of scanners that scan the top_port_i in each cluster from PCA_kMeans_clustering.
- Step 4: Save the resulted DataFrame in a list by converting it to an RDD, then use collect().  The list contains a list of ``( <cluster_id> , <number of scanners that scan top_port_i in the cluster> )``.

## Below is an example of Step 1, 2, and 3 for the OHE feature for the first top port.

In [112]:
i=0
feature_name= "Port" + Top_Ports_list[i]
feature_i_count_by_clusters = p_predictions.where(col(feature_name)).groupBy("pca_prediction").count()

In [113]:
fc_bc_rdd = feature_i_count_by_clusters.rdd

In [114]:
fc_bc_rdd.take(10)

[Row(pca_prediction=31, count=104),
 Row(pca_prediction=85, count=190),
 Row(pca_prediction=137, count=693),
 Row(pca_prediction=65, count=211),
 Row(pca_prediction=53, count=344),
 Row(pca_prediction=133, count=373),
 Row(pca_prediction=78, count=254),
 Row(pca_prediction=155, count=312),
 Row(pca_prediction=108, count=1004),
 Row(pca_prediction=193, count=70)]

## The DataFrame ``feature_i_count_by_clusters`` contains two columns: ``pca_prediction`` and ``count``.
## The RDD converted from the DataFrame has a Row object with these two columns, where ``count`` is the number of scanners in the ``pca_prediction`` cluster that scans a given top port (i.e., the first top port in this example).
## Can you explain what the first two elements of the RDD ``fc_bc_rdd`` mean?

## Naming convention: We use ``fc_bc`` in variables as a short hand notation for "feature count by cluster``.

In [115]:
fc_bc_list =fc_bc_rdd.collect()

In [116]:
print(fc_bc_list)

[Row(pca_prediction=31, count=104), Row(pca_prediction=85, count=190), Row(pca_prediction=137, count=693), Row(pca_prediction=65, count=211), Row(pca_prediction=53, count=344), Row(pca_prediction=133, count=373), Row(pca_prediction=78, count=254), Row(pca_prediction=155, count=312), Row(pca_prediction=108, count=1004), Row(pca_prediction=193, count=70), Row(pca_prediction=126, count=62), Row(pca_prediction=101, count=303), Row(pca_prediction=76, count=247), Row(pca_prediction=26, count=124), Row(pca_prediction=159, count=321), Row(pca_prediction=44, count=109), Row(pca_prediction=192, count=197), Row(pca_prediction=103, count=66), Row(pca_prediction=12, count=6), Row(pca_prediction=91, count=331), Row(pca_prediction=22, count=165), Row(pca_prediction=122, count=207), Row(pca_prediction=157, count=72), Row(pca_prediction=93, count=96), Row(pca_prediction=111, count=92), Row(pca_prediction=140, count=87), Row(pca_prediction=132, count=26), Row(pca_prediction=146, count=24), Row(pca_predi

# By iterating through ``fc_bc_list`` generated above, we can find the number of scanners that scan a given top port in each cluster generated by PCA_kmeans_clustering.

In [117]:
# For the first top port, the number of scanners in each PCA-generated cluster that scan the port.
for row in fc_bc_list:
    print("PCA_kmeans Cluster Index ", row[0], ": contains ", row[1], " scanners that scan Port ", Top_Ports_list[0] )

PCA_kmeans Cluster Index  31 : contains  104  scanners that scan Port  17132
PCA_kmeans Cluster Index  85 : contains  190  scanners that scan Port  17132
PCA_kmeans Cluster Index  137 : contains  693  scanners that scan Port  17132
PCA_kmeans Cluster Index  65 : contains  211  scanners that scan Port  17132
PCA_kmeans Cluster Index  53 : contains  344  scanners that scan Port  17132
PCA_kmeans Cluster Index  133 : contains  373  scanners that scan Port  17132
PCA_kmeans Cluster Index  78 : contains  254  scanners that scan Port  17132
PCA_kmeans Cluster Index  155 : contains  312  scanners that scan Port  17132
PCA_kmeans Cluster Index  108 : contains  1004  scanners that scan Port  17132
PCA_kmeans Cluster Index  193 : contains  70  scanners that scan Port  17132
PCA_kmeans Cluster Index  126 : contains  62  scanners that scan Port  17132
PCA_kmeans Cluster Index  101 : contains  303  scanners that scan Port  17132
PCA_kmeans Cluster Index  76 : contains  247  scanners that scan Port 

In [118]:
p_predictions.persist()
for i in range(0, top_k_ports):
    feature_i_name = "Port" + Top_Ports_list[i]
    feature_i_count_by_clusters_DF = p_predictions.where(col(feature_i_name)).groupBy(col("pca_prediction")).count()
    fic_bc_list = feature_i_count_by_clusters_DF.rdd.collect()
    # fic_bc_list is a list of (cluster_index, count of scanners that scan ith top port)
    for row in fic_bc_list:
        cluster_center_array[row[0]][i] = row[1]

25/04/07 19:02:35 WARN CacheManager: Asked to cache already cached data.


In [119]:
# total_clusters = 200

In [120]:
import pandas as pd

# Exercise 7 (25 points)
## Complete the code below for computing cluster centers for each cluster using the ``cluster_center_array`` calculated above.

In [121]:
# The number of total clusters (`total_clusters`) was specified earlier when we created k-means model template. 
# Define columns of the Pandas dataframe
column_list = ['cluster ID', 'size', 'mirai_ratio' ]
for feature in input_features:
    column_list.append(feature)
clusters_summary_df = pd.DataFrame( columns = column_list )
for i in range(0, total_clusters):
    cluster_i = p_predictions.where(col("pca_prediction") == i)
    cluster_i.persist()
    cluster_i_size = cluster_i.count()
    cluster_i_mirai_count = cluster_i.where(col("mirai")).count()
    cluster_i_mirai_ratio = cluster_i_mirai_count/cluster_i_size
    if cluster_i_mirai_count > 0:
        print("Cluster ", i, "; Mirai Ratio:", cluster_i_mirai_ratio, "; Cluster Size: ", cluster_i_size)
    cluster_row = [i, cluster_i_size, cluster_i_mirai_ratio]
    for j in range(0, len(input_features)):
        # compute the center for the original jth feature (i.e., jth top port)
        feature_j = "Port" + Top_Ports_list[j]
        count_i_j = cluster_center_array[i][j]
        # count_j = cluster_i.where(col(feature_j)).count()
        center_j = count_i_j / cluster_i_size
        cluster_row.append(center_j)
    clusters_summary_df.loc[i]= cluster_row
    cluster_i.unpersist()

Cluster  2 ; Mirai Ratio: 0.024424284717376135 ; Cluster Size:  1433


Cluster  10 ; Mirai Ratio: 0.004997858060831073 ; Cluster Size:  7003
Cluster  16 ; Mirai Ratio: 0.031062124248496994 ; Cluster Size:  998
Cluster  28 ; Mirai Ratio: 0.011456628477905073 ; Cluster Size:  611
Cluster  29 ; Mirai Ratio: 0.016180371352785147 ; Cluster Size:  7540
Cluster  43 ; Mirai Ratio: 0.8665086887835703 ; Cluster Size:  1266
Cluster  83 ; Mirai Ratio: 0.01699029126213592 ; Cluster Size:  412
Cluster  87 ; Mirai Ratio: 0.003745318352059925 ; Cluster Size:  267


In [122]:
path5= "/storage/home/ajv5723/work/MiniProj3/local/Mirai_Ratio_120OHE_PCA30_k200.csv"
clusters_summary_df.to_csv(path5, header=True)

In [123]:
clusters_summary_df

,cluster ID,size,mirai_ratio,Port17132,Port17130,Port17140,Port17128,Port17138,Port17136,Port17134,...,Port9943,Port143,Port3000,Port9080,Port10000,Port17,Port135,Port587,Port993,Port9999
0,0.0,410.0,0.000000,1.000000,0.000000,0.000000,0.231707,0.000000,0.000000,1.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1.0,53.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.0,1.000000,1.000000,1.000000,1.000000,0.981132,0.867925,1.000000,1.000000,1.000000
2,2.0,1433.0,0.024424,0.000698,0.000000,0.000698,0.000698,0.001396,0.000000,0.000000,...,0.0,0.002094,0.002094,0.001396,0.001396,0.000000,0.003489,0.002094,0.000698,0.004187
3,3.0,377.0,0.000000,1.000000,1.000000,0.416446,0.000000,1.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,4.0,375.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,195.0,176.0,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
196,196.0,44.0,0.000000,0.272727,0.818182,0.863636,0.886364,1.000000,0.136364,0.022727,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
197,197.0,631.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.028526,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
198,198.0,135.0,0.000000,0.933333,0.977778,0.911111,0.992593,0.874074,0.948148,0.807407,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# Exercise 8 (40 points)
Modify the Jupyter Notebook for running in cluster mode using the big dataset (Day_2020_profile.csv). Make sure you change the output directory from `../local/..` to
`../cluster/..` so that it does not destroy the result you obtained in local mode.
Run the .py file the cluster mode to calculate cluster centers and Mirai percentage for each cluster with and without PCA.
- Submit the .py file  (5 points)
- Submit the the log file that contains the run time information for a successful execution in the cluster mode. (5 points)
- Submit the output file that records the cluster summary in the cluster mode (without PCA) (10 points)
- Submit the output file that records the cluster summary in the cluster mode (with PCA) (10 points)
- Discuss the Silihouette score and Mirai ratio of clusters generated by k-means clustering with PCA and without PCA (in a separate word document) (10 points)

In [124]:
ss.stop()